# Notebook 17 — MobileBERT Zero-Shot Evaluation (EC2 g4dn.xlarge)

## Environment
- **Instance**: AWS EC2 g4dn.xlarge
- **GPU**: NVIDIA T4 (16GB VRAM)
- **Framework**: PyTorch + HuggingFace Transformers

## Objective
Evaluate `cssupport/mobilebert-sql-injection-detect` (zero-shot) against the
SQLi-only subset of CSIC 2010.

## Why zero-shot only
Fine-tuning `google/mobilebert-uncased` was attempted across NB15, NB16, and NB17
but produced exploding losses (epoch 1 loss ~1M–3M) across all three input strategies,
resulting in collapsed models. Root cause: loading a masked LM checkpoint for sequence
classification produces extreme logits before training. The zero-shot result from
`cssupport/mobilebert-sql-injection-detect` — a properly fine-tuned classifier — is
the valid and meaningful MobileBERT result.

## Model
| Model | Source | Training data |
|---|---|---|
| `cssupport/mobilebert-sql-injection-detect` | HuggingFace Hub | `Modified_SQL_Dataset.csv` (Kaggle) |

## Dataset
- **Positive**: 1,389 SQLi-only entries from CSIC 2010 `anomalousTrafficTest.txt`
- **Negative**: 56,000 benign entries from CSIC 2010
- SQLi entries verified by manual review (n=130) — zero SQLi in excluded entries

## 0. Install Dependencies

In [1]:
import subprocess
subprocess.run(['pip', 'install', '-q',
                'transformers', 'torch', 'scikit-learn',
                'pandas', 'numpy', 'accelerate'])
print('Dependencies installed.')

Dependencies installed.


## 1. Imports & Paths

In [2]:
import os
import re
import time
import json
import random
import shutil
import urllib.parse
from collections import Counter

import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import (
    MobileBertTokenizer,
    MobileBertForSequenceClassification,
    get_linear_schedule_with_warmup,
)

# ── Paths ─────────────────────────────────────────────────────────
BASE         = os.path.expanduser('~/webattack-detector')
RESULTS_DIR  = f'{BASE}/notebooks/results/metrics'
MODELS_DIR   = f'{BASE}/notebooks/results/models'
TRAINING_DIR = f'{BASE}/notebooks/results/training_data'

ATTACK_FILE    = f'{BASE}/logs/csic2010/anomalousTrafficTest.txt'
BENIGN_TEST    = f'{BASE}/logs/csic2010/normalTrafficTest.txt'
BENIGN_TRAIN   = f'{BASE}/logs/csic2010/normalTrafficTraining.txt'
POSITIVES_PATH = f'{TRAINING_DIR}/07_positives.txt'
NEGATIVES_PATH = f'{TRAINING_DIR}/07_negatives.txt'

for d in [RESULTS_DIR, MODELS_DIR, TRAINING_DIR]:
    os.makedirs(d, exist_ok=True)

# ── Model config ──────────────────────────────────────────────────
BASE_MODEL_NAME     = 'google/mobilebert-uncased'
ZEROSHOT_MODEL_NAME = 'cssupport/mobilebert-sql-injection-detect'
MAX_LENGTH = 128
BATCH_SIZE = 128
DEVICE     = 'cuda' if torch.cuda.is_available() else 'cpu'

random.seed(42)
torch.manual_seed(42)

# ── GPU info ──────────────────────────────────────────────────────
print(f'Device : {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

# ── Verify files ──────────────────────────────────────────────────
print()
all_ok = True
for label, path in [
    ('Attack file',  ATTACK_FILE),
    ('Benign test',  BENIGN_TEST),
    ('Benign train', BENIGN_TRAIN),
    ('Positives',    POSITIVES_PATH),
    ('Negatives',    NEGATIVES_PATH),
]:
    exists = os.path.exists(path)
    size   = os.path.getsize(path) // 1024 if exists else 0
    status = '✅' if exists else '❌'
    print(f'  {status} {label:<14}: {size:>6} KB')
    if not exists:
        all_ok = False

print('\nAll files found. Ready.' if all_ok else '\n⚠️  Some files missing.')

Device : cuda
GPU    : Tesla T4
VRAM   : 15.6 GB

  ✅ Attack file   :  15713 KB
  ✅ Benign test   :  20116 KB
  ✅ Benign train  :  20157 KB
  ✅ Positives     :   1339 KB
  ✅ Negatives     :     20 KB

All files found. Ready.


## 2. CSIC 2010 Parser

Fixed parser that correctly separates HTTP headers from POST body.
Handles three separator styles found across the three CSIC files:
- Attack file + normalTrafficTraining: triple CRLF between requests
- normalTrafficTest: double CRLF between requests

In [3]:
HTTP_METHODS = ('GET', 'POST', 'PUT', 'DELETE', 'HEAD', 'OPTIONS')

def parse_csic_file(filepath, label):
    entries = []
    with open(filepath, 'rb') as f:
        raw = f.read()

    if b'\r\n\r\n\r\n' in raw:
        request_sep = b'\r\n\r\n\r\n'
        header_sep  = b'\r\n\r\n'
        line_sep    = '\r\n'
    elif b'\r\n\r\n' in raw:
        request_sep = b'\r\n\r\n'
        header_sep  = None
        line_sep    = '\r\n'
    else:
        request_sep = b'\n\n\n'
        header_sep  = b'\n\n'
        line_sep    = '\n'

    def clean_body(body_text, ls):
        lines = body_text.split(ls)
        clean = []
        for line in lines:
            stripped = line.strip()
            if any(stripped.startswith(m + ' ') for m in HTTP_METHODS) \
                    and 'HTTP/' in stripped:
                break
            clean.append(line)
        return ls.join(clean).strip()

    def process_block(block_bytes):
        block = block_bytes.strip()
        if not block:
            return None
        if b'\r\n\r\n' in block:
            hdr_bytes, _, body_bytes = block.partition(b'\r\n\r\n')
            ls = '\r\n'
        elif b'\n\n' in block:
            hdr_bytes, _, body_bytes = block.partition(b'\n\n')
            ls = '\n'
        else:
            hdr_bytes  = block
            body_bytes = b''
            ls         = '\r\n'

        header_text = hdr_bytes.decode('latin-1', errors='ignore')
        body_raw    = body_bytes.decode('latin-1', errors='ignore').strip()
        body_text   = clean_body(body_raw, ls)

        lines = header_text.split(ls)
        first = lines[0].strip()
        parts = first.split(' ')

        if len(parts) < 2:
            return None
        if parts[0].upper() not in HTTP_METHODS:
            return None
        if 'HTTP/' not in first:
            return None

        try:
            parsed = urllib.parse.urlparse(parts[1])
        except Exception:
            return None

        post_params = {}
        if parts[0].upper() == 'POST' and body_text:
            try:
                first_body_line = body_text.split(ls)[0].strip()
                post_params = urllib.parse.parse_qs(
                    first_body_line, keep_blank_values=True
                )
            except Exception:
                pass

        return {
            'method':      parts[0].upper(),
            'url':         parts[1],
            'protocol':    parts[2] if len(parts) > 2 else 'HTTP/1.1',
            'request_line': first,
            'path':        parsed.path,
            'qs':          parsed.query,
            'body':        body_text,
            'post_params': post_params,
            'label':       label,
        }

    if request_sep == b'\r\n\r\n' and header_sep is None:
        chunks  = raw.split(b'\r\n\r\n')
        current = b''
        for chunk in chunks:
            chunk = chunk.strip()
            if not chunk:
                continue
            decoded    = chunk.decode('latin-1', errors='ignore')
            first_line = decoded.split('\r\n')[0].strip()
            fparts     = first_line.split(' ')
            if (len(fparts) >= 2
                    and fparts[0].upper() in HTTP_METHODS
                    and 'HTTP/' in first_line):
                if current:
                    entry = process_block(current)
                    if entry:
                        entries.append(entry)
                current = chunk
            else:
                current = current + b'\r\n\r\n' + chunk
        if current:
            entry = process_block(current)
            if entry:
                entries.append(entry)
    else:
        for raw_req in raw.split(request_sep):
            entry = process_block(raw_req)
            if entry:
                entries.append(entry)

    print(f'  {filepath.split("/")[-1]}: {len(entries):,} entries')
    return entries


print('Loading CSIC 2010...')
attack_entries = parse_csic_file(ATTACK_FILE,  label=1)
benign_test    = parse_csic_file(BENIGN_TEST,   label=0)
benign_train   = parse_csic_file(BENIGN_TRAIN,  label=0)
benign_entries = benign_test + benign_train

print(f'\nAttack : {len(attack_entries):,}')
print(f'Benign : {len(benign_entries):,}')

# POST body sanity check
print('\nPOST body check (first 3 POST attacks):')
for e in [e for e in attack_entries if e['method'] == 'POST'][:3]:
    print(f'  Body: {urllib.parse.unquote(e["body"][:100])}')

Loading CSIC 2010...
  anomalousTrafficTest.txt: 15,088 entries
  normalTrafficTest.txt: 28,000 entries
  normalTrafficTraining.txt: 28,000 entries

Attack : 15,088
Benign : 56,000

POST body check (first 3 POST attacks):
  Body: id=2&nombre=Jam�n+Ib�rico&precio=85&cantidad=';+DROP+TABLE+usuarios;+SELECT+*+FROM+datos+W
  Body: id=2/&nombre=Jam�n+Ib�rico&precio=85&cantidad=49&B1=A�adir+al+carrito
  Body: modo=entrar&login=bob@<SCRipt>alert(Paros)</scrIPT>.parosproxy.org&pwd=84m3ri156&rem


## 3. Attack Type Classifier

21 SQL pattern families cover all major SQLi variants:
UNION-based, boolean-based, time-based, stacked queries, DDL injection,
string comparison bypasses, and encoding variants.

Additional patterns cover XSS, SSI, Header Injection, Path Traversal,
Buffer Overflow, Null Byte, Reconnaissance, and Parameter Tampering.

In [4]:
SQL_PATTERNS = [
    r"(?i)(union[\s\+%09%0a%0d]+select)",
    r"(?i)(select[\s\+%09]+.{0,40}[\s\+%09]+from)",
    r"(?i)(insert[\s\+%09]+into)",
    r"(?i)(update[\s\+%09]+\w+[\s\+%09]+set)",
    r"(?i)(delete[\s\+%09]+from)",
    r"(?i)(drop[\s\+%09]+(table|database|schema))",
    r"(?i)(alter[\s\+%09]+(table|database))",
    r"(?i)(create[\s\+%09]+(table|user|database))",
    r"(?i)(or[\s\+%09]+[\'\"]?1[\'\"]?[\s\+%09]*=[\s\+%09]*[\'\"]?1)",
    r"(?i)(and[\s\+%09]+[\'\"]?1[\'\"]?[\s\+%09]*=[\s\+%09]*[\'\"]?1)",
    r"(?i)(or[\s\+%09]+['\"][^'\"]{0,10}['\"][\s\+%09]*=[\s\+%09]*['\"][^'\"]{0,10}['\"])",
    r"(?i)(sleep[\s\+%09]*\()",
    r"(?i)(benchmark[\s\+%09]*\()",
    r"(?i)(waitfor[\s\+%09]+delay)",
    r"(?i)(information_schema)",
    r"(?i)(load_file[\s\+%09]*\()",
    r"(?i)(into[\s\+%09]+(outfile|dumpfile))",
    r"(?i)(pg_sleep[\s\+%09]*\()",
    r"(?i)(xp_cmdshell)",
    r"(?i)('[\s\+%09]*;[\s\+%09]*(select|insert|update|delete|drop))",
    r"(?i)(--[\s\+%09]*$|#[\s\+%09]*$|/\*.*?\*/)",
    r"(';|';--|' --)",
]

XSS_PATTERNS = [
    r"(?i)(<script[\s>])", r"(?i)(javascript[\s\+%09]*:)",
    r"(?i)(onerror[\s\+%09]*=)", r"(?i)(onload[\s\+%09]*=)",
    r"(?i)(alert[\s\+%09]*\()", r"(?i)(<iframe[\s>])",
    r"(?i)(document\.cookie)", r"(?i)(eval[\s\+%09]*\()",
    r"(?i)(%3cscript)", r"(?i)(vbscript[\s\+%09]*:)",
]

PATH_TRAVERSAL_PATTERNS = [
    r"(\.\./)|(\.\.\\)",
    r"(%2e%2e%2f|%2e%2e/|\.\.\.%2f)",
    r"(%252e%252e)",
    r"(etc/passwd|etc/shadow|win\.ini|boot\.ini)",
]

BUFFER_OVERFLOW_PATTERNS = [
    r"(A{50,}|%41{50,})",
    r"(.{200,})",
]

SSI_PATTERNS = [
    r"(?i)(<!--\s*#\s*(include|exec|echo|printenv|set)\s)",
    r"(?i)(<!--#)",
]

HEADER_INJECTION_PATTERNS = [
    r"(?i)(set-cookie\s*:)",
    r"(?i)(%0d%0a|%0a%0d).*:",
    r"(?i)(%0[aA]Set-cookie)",
    r"(?i)(%0[dD]%0[aA]Set-cookie)",
    r"(?i)(%3[fF]%0[dD]%0[aA])",
    r"(?i)(%0[aA][A-Za-z\-]+:)",
]

NULL_BYTE_PATTERNS     = [r"(%00|%2500|\x00)"]
CSTI_PATTERNS          = [r"(?i)(csrf|xsrf)", r"(?i)(__requestverificationtoken)"]

RECON_PATTERNS = [
    r"(?i)\.(bak|old|inc|backup|orig|tmp|swp|~)$",
    r"(?i)\.(bak|old|inc|backup|orig|tmp|swp|~)[/.]",
    r"(?i)(IISSamples|_cti_pvt|_vti_|webcart|scripts/tools|webapp/examples)",
    r"(?i)\d{10,}\.(java|old|bak|jsp|gif|jpg)",
    r"(?i)localhost:\d+\.(old|OLD|java|bak)$",
    r"(?i)\.(Old|OLD|BAK|Inc|INC|java|gif\.java)$",
    r"(?i)/servlet/",
]

PARAMETER_TAMPERING_PATTERNS = [
    r"(?i)([a-z]+A=)",
    r"(%252[Bb]|%252F|%253F)",
    r"(%2500)",
]

INJECTED_PARAM_PATTERNS    = [r"'INJECTED_PARAM", r"INJECTED_PARAM"]
REGEX_INJECTION_PATTERNS   = [r"(\.\*\?|\.\+|\[\^|\(\?:)", r"(\.\*|\\\w|\(\.\))"]

VALUE_TAMPERING_PATTERNS = [
    r"(precio=\d+[%+/|?&])", r"(cantidad=\d+[%+/|?&])",
    r"(id=\d+[/|?&])",       r"(B1=.*[/|?&][^&]*$)",
    r"(B2=.*[/|?&][^&]*$)",  r"(\|$|^\|)",
]


def classify_attack(entry):
    url_decoded  = urllib.parse.unquote(entry.get('url', ''))
    qs_decoded   = urllib.parse.unquote(entry.get('qs', ''))
    body_decoded = urllib.parse.unquote(
        entry.get('body', '').split('\r\n')[0].split('\n')[0]
    )
    text = f"{url_decoded} {qs_decoded} {body_decoded}"

    types = []
    if any(re.search(p, text) for p in SQL_PATTERNS):               types.append('SQLi')
    if any(re.search(p, text) for p in XSS_PATTERNS):               types.append('XSS')
    if any(re.search(p, text) for p in SSI_PATTERNS):               types.append('SSI Injection')
    if any(re.search(p, text) for p in HEADER_INJECTION_PATTERNS):  types.append('Header Injection')
    if any(re.search(p, text) for p in NULL_BYTE_PATTERNS):         types.append('Null Byte')
    if any(re.search(p, text) for p in PATH_TRAVERSAL_PATTERNS):    types.append('Path Traversal')
    if any(re.search(p, text) for p in BUFFER_OVERFLOW_PATTERNS):   types.append('Buffer Overflow')
    if any(re.search(p, text) for p in CSTI_PATTERNS):              types.append('CSTI/XSRF')
    if any(re.search(p, text) for p in RECON_PATTERNS):             types.append('Reconnaissance')
    if any(re.search(p, text) for p in PARAMETER_TAMPERING_PATTERNS): types.append('Parameter Tampering')
    if any(re.search(p, text) for p in INJECTED_PARAM_PATTERNS):    types.append('Injected Param')
    if any(re.search(p, text) for p in REGEX_INJECTION_PATTERNS):   types.append('Regex Injection')
    if any(re.search(p, text) for p in VALUE_TAMPERING_PATTERNS):   types.append('Value Tampering')
    if entry.get('method') in ('PUT', 'DELETE') and '/tienda1/' in entry.get('url', ''):
        types.append('HTTP Method Abuse')

    return types if types else ['Unclassified']


# Secondary classifier for remaining unclassified
def classify_unclassified(entry):
    url_decoded  = urllib.parse.unquote(entry.get('url', ''))
    qs_decoded   = urllib.parse.unquote(entry.get('qs', ''))
    body_decoded = urllib.parse.unquote(
        entry.get('body', '').split('\r\n')[0].split('\n')[0]
    )
    text = f"{url_decoded} {qs_decoded} {body_decoded}"

    if re.search(r"(?i)\.(bak|old|inc|backup|orig|tmp|swp|~|Old|OLD|BAK|Inc|INC|java)(\b|$|/|\?)", url_decoded) or \
       re.search(r"(?i)(servlet|showcfg|drvrs|IISSamples|_cti|asf-logo|\d{10,})", url_decoded):
        return ['Reconnaissance']
    if entry.get('method') in ('PUT', 'DELETE', 'HEAD', 'OPTIONS'):
        return ['HTTP Method Abuse']
    if re.search(r"(\.\*\?|\.\+|\[\^|\\[dDwWsS]|\(\?)", text):
        return ['Regex Injection']
    if re.search(r"(=[^&]*[|@?%+][^&]*(&|$))", qs_decoded + body_decoded):
        return ['Value Tampering']
    if re.search(r"(=[^&]*[/][^&]*(&|$))", qs_decoded + body_decoded):
        return ['Value Tampering']
    if entry.get('method') == 'POST':
        return ['Parameter Tampering']
    return ['Reconnaissance']


# Classify all entries
print('Classifying attack types...')
for e in attack_entries:
    e['attack_types'] = classify_attack(e)

# Resolve unclassified
resolved = sum(1 for e in attack_entries if e['attack_types'] == ['Unclassified'])
for e in attack_entries:
    if e['attack_types'] == ['Unclassified']:
        e['attack_types'] = classify_unclassified(e)

total          = len(attack_entries)
primary_counts = Counter(e['attack_types'][0] for e in attack_entries)
multi_counts   = Counter(t for e in attack_entries for t in e['attack_types'])
unclassified   = primary_counts.get('Unclassified', 0)

print(f'\nTotal attack entries : {total:,}')
print(f'Resolved unclassified: {resolved:,}')
print(f'Remaining unclassified: {unclassified}')
print()
print('─' * 58)
print(f'{"Attack Type":<36} {"Count":>7}  {"% of Total":>10}')
print('─' * 58)
for atype, count in primary_counts.most_common():
    pct = count / total * 100
    bar = '█' * int(pct / 2)
    print(f'{atype:<36} {count:>7,}  {pct:>9.1f}%  {bar}')
print('─' * 58)

Classifying attack types...

Total attack entries : 15,088
Resolved unclassified: 4,093
Remaining unclassified: 0

──────────────────────────────────────────────────────────
Attack Type                            Count  % of Total
──────────────────────────────────────────────────────────
Value Tampering                        3,257       21.6%  ██████████
Reconnaissance                         3,166       21.0%  ██████████
Buffer Overflow                        2,685       17.8%  ████████
Parameter Tampering                    2,236       14.8%  ███████
SQLi                                   1,389        9.2%  ████
XSS                                      811        5.4%  ██
Header Injection                         653        4.3%  ██
SSI Injection                            510        3.4%  █
Null Byte                                121        0.8%  
Injected Param                           115        0.8%  
HTTP Method Abuse                         88        0.6%  
Regex Injection  

## 4. SQLi Subset Extraction

Extract entries confirmed as SQL injection and verify the subset quality.

In [5]:
sqli_entries = [e for e in attack_entries if 'SQLi' in e['attack_types']]
non_sqli     = [e for e in attack_entries if 'SQLi' not in e['attack_types']]

print('=' * 60)
print('SQLi SUBSET SUMMARY')
print('=' * 60)
print(f'Total CSIC attack entries  : {len(attack_entries):,}')
print(f'SQLi entries (positive)    : {len(sqli_entries):,}  '
      f'({len(sqli_entries)/len(attack_entries)*100:.1f}% of attacks)')
print(f'Non-SQLi attacks (excluded): {len(non_sqli):,}')
print(f'Benign entries (negative)  : {len(benign_entries):,}')
print()

# Verification 1: all SQLi confirmed by pattern
false_sqli = []
for e in sqli_entries:
    url_decoded  = urllib.parse.unquote(e.get('url', ''))
    qs_decoded   = urllib.parse.unquote(e.get('qs', ''))
    body_decoded = urllib.parse.unquote(
        e.get('body', '').split('\r\n')[0].split('\n')[0]
    )
    text = f"{url_decoded} {qs_decoded} {body_decoded}"
    if not any(re.search(p, text) for p in SQL_PATTERNS):
        false_sqli.append(e)

print('─' * 60)
print('VERIFICATION')
print('─' * 60)
status = '✅' if len(false_sqli) == 0 else '❌'
print(f'{status} Pattern confirmation: {len(sqli_entries)-len(false_sqli):,} / {len(sqli_entries):,} '
      f'entries confirmed by SQL pattern')

# Method breakdown
method_counts = Counter(e['method'] for e in sqli_entries)
print()
print('SQLi by HTTP method:')
for method, count in method_counts.most_common():
    pct = count / len(sqli_entries) * 100
    print(f'  {method:<8} {count:>5,}  ({pct:.1f}%)')

# Sample SQLi entries
print()
print('─' * 60)
print('Sample SQLi payloads (10 random):')
print('─' * 60)
random.seed(42)
for i, e in enumerate(random.sample(sqli_entries, 10), 1):
    qs   = urllib.parse.unquote(e.get('qs', ''))[:90]
    body = urllib.parse.unquote(
        e.get('body', '').split('\r\n')[0].split('\n')[0]
    )[:90]
    payload = qs if qs else body
    print(f'[{i:>2}] {e["method"]} | {payload}')

SQLi SUBSET SUMMARY
Total CSIC attack entries  : 15,088
SQLi entries (positive)    : 1,389  (9.2% of attacks)
Non-SQLi attacks (excluded): 13,699
Benign entries (negative)  : 56,000

────────────────────────────────────────────────────────────
VERIFICATION
────────────────────────────────────────────────────────────
✅ Pattern confirmation: 1,389 / 1,389 entries confirmed by SQL pattern

SQLi by HTTP method:
  POST       987  (71.1%)
  GET        402  (28.9%)

────────────────────────────────────────────────────────────
Sample SQLi payloads (10 random):
────────────────────────────────────────────────────────────
[ 1] GET | modo=');waitfor+delay+'0:0:15';--&login=linebarg&password=6A6ejo&nombre=Adiberto&apellidos
[ 2] POST | modo=entrar&login=zakarow','0','0');waitfor+delay+'0:0:15';--&pwd=e4pe5imenta2a&remember=o
[ 3] GET | modo=entrar&login=dadang','0','0');waitfor+delay+'0:0:15';--&pwd=6a1er87&remember=on&B1=En
[ 4] POST | modo=insertar','0','0','0');waitfor+delay+'0:0:15';--&precio=

## 5. Helpers

Minimal helper set — only what the zero-shot evaluation requires:
`load_zeroshot`, `score_batched`, `threshold_scan`, `best_threshold_fine`.

In [6]:
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    MobileBertTokenizer,
    MobileBertForSequenceClassification,
)

BASE_MODEL_NAME      = 'google/mobilebert-uncased'
ZEROSHOT_MODEL_NAME  = 'cssupport/mobilebert-sql-injection-detect'
MAX_LENGTH = 128
BATCH_SIZE = 128
DEVICE     = 'cuda' if torch.cuda.is_available() else 'cpu'

print(f'Device : {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')


def extract_query_values(entry):
    """Query string values for GET; POST body values as fallback."""
    try:
        qs = entry.get('qs', '')
        if qs:
            params = urllib.parse.parse_qs(qs, keep_blank_values=False)
            values = [
                urllib.parse.unquote(v).strip()
                for vlist in params.values()
                for v in vlist
                if urllib.parse.unquote(v).strip()
            ]
            if values:
                return ' '.join(values)
        body = entry.get('body', '').split('\r\n')[0].split('\n')[0].strip()
        if body:
            params = urllib.parse.parse_qs(body, keep_blank_values=False)
            values = [
                urllib.parse.unquote(v).strip()
                for vlist in params.values()
                for v in vlist
                if urllib.parse.unquote(v).strip()
            ]
            if values:
                return ' '.join(values)
        return None
    except Exception:
        return None


def score_batched(entries, extract_fn, model, tokenizer,
                  batch_size=128, device='cpu', label=''):
    texts, indices = [], []
    for i, e in enumerate(entries):
        text = extract_fn(e)
        if text and text.strip():
            texts.append(text)
            indices.append(i)
    print(f'  {label} scoreable: {len(texts):,} / {len(entries):,}')
    if not texts:
        return [0.0] * len(entries)
    all_probs = []
    model.eval()
    model.to(device)
    n_batches = (len(texts) - 1) // batch_size + 1
    for i in range(0, len(texts), batch_size):
        batch  = texts[i:i+batch_size]
        inputs = tokenizer(batch, truncation=True, padding=True,
                           max_length=MAX_LENGTH, return_tensors='pt').to(device)
        with torch.no_grad():
            probs = torch.softmax(model(**inputs).logits, dim=1)[:, 1].cpu().numpy()
        all_probs.extend(probs.tolist())
        batch_num = i // batch_size + 1
        if batch_num % 50 == 0 or batch_num == n_batches:
            print(f'    Batch {batch_num}/{n_batches} ({batch_num/n_batches*100:.0f}%)', flush=True)
    result = [0.0] * len(entries)
    for idx, prob in zip(indices, all_probs):
        result[idx] = prob
    return result


def threshold_scan(label, probs_a, probs_b):
    print(f'\n{"="*70}')
    print(f'MobileBERT | {label}')
    print(f'{"="*70}')
    print(f'{"Threshold":>12} {"Recall":>8} {"FP/10k":>8} '
          f'{"Precision":>10} {"Detected":>10} {"F1":>8}')
    print('-' * 65)
    best = None
    for t in [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95, 0.99]:
        detected  = sum(1 for s in probs_a if s >= t)
        fp        = sum(1 for s in probs_b if s >= t)
        recall    = detected / len(probs_a) if probs_a else 0
        fp_10k    = (fp / len(probs_b)) * 10000 if probs_b else 0
        precision = detected / (detected + fp) if (detected + fp) > 0 else 0
        f1        = (2 * precision * recall / (precision + recall)) \
                    if (precision + recall) > 0 else 0
        marker    = ' <-' if (best is None or f1 > best['f1']) else ''
        print(f'{t:>12.2f} {recall:>8.4f} {fp_10k:>8.2f} '
              f'{precision:>10.4f} {detected:>10,} {f1:>8.4f}{marker}')
        if best is None or f1 > best['f1']:
            best = {'threshold': t, 'recall': recall, 'fp_10k': fp_10k,
                    'precision': precision, 'f1': f1}
    print(f'\n  Best F1 @ threshold {best["threshold"]}: '
          f'recall={best["recall"]:.4f}, fp_10k={best["fp_10k"]:.2f}, '
          f'f1={best["f1"]:.4f}')
    return best


def best_threshold_fine(probs_a, probs_b):
    best = None
    for t in np.arange(0.3, 1.0, 0.01):
        detected  = sum(1 for s in probs_a if s >= t)
        fp        = sum(1 for s in probs_b if s >= t)
        recall    = detected / len(probs_a) if probs_a else 0
        precision = detected / (detected + fp) if (detected + fp) > 0 else 0
        f1        = (2 * precision * recall / (precision + recall)) \
                    if (precision + recall) > 0 else 0
        fp_10k    = (fp / len(probs_b)) * 10000 if probs_b else 0
        if best is None or f1 > best['f1']:
            best = {'t': round(float(t), 2), 'recall': recall,
                    'fp_10k': fp_10k, 'precision': precision, 'f1': f1}
    return best


def load_zeroshot(device):
    local_path = f'{MODELS_DIR}/15_mobilebert_zeroshot'
    if os.path.exists(local_path):
        print(f'Loading from cache: {local_path}')
        tokenizer = MobileBertTokenizer.from_pretrained('google/mobilebert-uncased')
        model     = MobileBertForSequenceClassification.from_pretrained(local_path)
    else:
        print(f'Downloading {ZEROSHOT_MODEL_NAME} from HuggingFace Hub...')
        tokenizer = MobileBertTokenizer.from_pretrained('google/mobilebert-uncased')
        model     = MobileBertForSequenceClassification.from_pretrained(ZEROSHOT_MODEL_NAME)
        os.makedirs(local_path, exist_ok=True)
        model.save_pretrained(local_path)
        print(f'  Cached: {local_path}')
    model.to(device)
    model.eval()
    return model, tokenizer


print('Helpers defined. Ready to evaluate.')


Device : cuda
GPU    : Tesla T4
VRAM   : 15.6 GB
Helpers defined. Ready to evaluate.


## 6. Zero-Shot Evaluation

**Model**: `cssupport/mobilebert-sql-injection-detect` — loaded from HuggingFace Hub as-is.

**Strategy**: Query values only — the closest match to the model's training distribution
(trained on bare SQL query strings from `Modified_SQL_Dataset.csv`).

**Note on scoreable count**: `extract_query_values` extracts parameter values from the
query string (GET) or POST body. Entries with no query string and no body (e.g. path-only
requests) return `None` and score 0.0 by default — they cannot generate false positives.
This affects 30,000 of the 56,000 benign entries and all 1,389 SQLi entries are scoreable
(SQLi payloads always appear in query string or POST body).

In [7]:
print('Loading zero-shot model...')
model_zs, tok_zs = load_zeroshot(DEVICE)

print('\nScoring CSIC 2010 SQLi subset...')
t0 = time.perf_counter()
probs_a_zs = score_batched(sqli_entries,   extract_query_values, model_zs, tok_zs, BATCH_SIZE, DEVICE, 'SQLi')
probs_b_zs = score_batched(benign_entries, extract_query_values, model_zs, tok_zs, BATCH_SIZE, DEVICE, 'Benign')
elapsed_zs = time.perf_counter() - t0
print(f'Done in {elapsed_zs:.1f}s')

a, b = np.array(probs_a_zs), np.array(probs_b_zs)
print(f'\nScore distributions:')
print(f'  SQLi   — mean: {a.mean():.3f}  min: {a.min():.3f}  max: {a.max():.3f}')
print(f'  Benign — mean: {b.mean():.3f}  min: {b.min():.3f}  max: {b.max():.3f}')

best_zs = threshold_scan('Zero-shot | Query Values (cssupport)', probs_a_zs, probs_b_zs)

with open(f'{RESULTS_DIR}/17_probs_zeroshot.json', 'w') as f:
    json.dump({'sqli': probs_a_zs, 'benign': probs_b_zs}, f)
print('\nProbabilities saved.')

del model_zs; torch.cuda.empty_cache()
print('Model unloaded from GPU.')

Loading zero-shot model...
Loading from cache: /home/ubuntu/webattack-detector/notebooks/results/models/15_mobilebert_zeroshot


Loading weights:   0%|          | 0/1113 [00:00<?, ?it/s]


Scoring CSIC 2010 SQLi subset...
  SQLi scoreable: 1,389 / 1,389
    Batch 11/11 (100%)
  Benign scoreable: 26,000 / 56,000
    Batch 50/204 (25%)
    Batch 100/204 (49%)
    Batch 150/204 (74%)
    Batch 200/204 (98%)
    Batch 204/204 (100%)
Done in 39.5s

Score distributions:
  SQLi   — mean: 0.821  min: 0.001  max: 1.000
  Benign — mean: 0.032  min: 0.000  max: 1.000

MobileBERT | Zero-shot | Query Values (cssupport)
   Threshold   Recall   FP/10k  Precision   Detected       F1
-----------------------------------------------------------------
        0.30   0.8215   300.00     0.4045      1,141   0.5420 <-
        0.40   0.8193   287.86     0.4138      1,138   0.5499 <-
        0.50   0.8186   276.61     0.4233      1,137   0.5580 <-
        0.60   0.8186   266.61     0.4323      1,137   0.5658 <-
        0.70   0.8186   257.32     0.4410      1,137   0.5732 <-
        0.80   0.8164   247.32     0.4502      1,134   0.5803 <-
        0.90   0.8157   236.07     0.4615      1,133   0

## 7. Results Summary

In [8]:
# Reload from disk if kernel restarted
def load_probs(path):
    with open(path) as f:
        d = json.load(f)
    return d['sqli'], d['benign']

if 'probs_a_zs' not in dir():
    probs_a_zs, probs_b_zs = load_probs(f'{RESULTS_DIR}/17_probs_zeroshot.json')
    print('Loaded zero-shot probs from disk.')

bzs = best_threshold_fine(probs_a_zs, probs_b_zs)

W = 90
print('=' * W)
print('MobileBERT ZERO-SHOT RESULTS — SQLi-only subset of CSIC 2010')
print('1,389 SQLi attacks vs 56,000 benign | Best F1 (fine-grained scan, step=0.01)')
print('=' * W)
print(f'{"Model":<14} {"Mode":<12} {"Strategy":<24} {"Thresh":>7} '
      f'{"Recall":>8} {"FP/10k":>8} {"Precision":>10} {"F1":>8}')
print('-' * W)
print(f'{"MobileBERT":<14} {"zero-shot":<12} {"Way 3 (cssupport)":<24} {bzs["t"]:>7.2f} '
      f'{bzs["recall"]:>8.4f} {bzs["fp_10k"]:>8.2f} '
      f'{bzs["precision"]:>10.4f} {bzs["f1"]:>8.4f}')
print('=' * W)

summary = [{
    'model': 'MobileBERT', 'mode': 'zero-shot',
    'strategy': 'Way 3 (cssupport)', 'threshold': bzs['t'],
    'recall': round(bzs['recall'], 4), 'fp_10k': round(bzs['fp_10k'], 4),
    'precision': round(bzs['precision'], 4), 'f1': round(bzs['f1'], 4),
    'notebook': 'NB17',
}]

pd.DataFrame(summary).to_csv(f'{RESULTS_DIR}/17_mobilebert_sqli_only_summary.csv', index=False)
with open(f'{RESULTS_DIR}/17_mobilebert_sqli_only_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
print(f'\nSaved: {RESULTS_DIR}/17_mobilebert_sqli_only_summary.csv')
print(f'Saved: {RESULTS_DIR}/17_mobilebert_sqli_only_summary.json')


MobileBERT ZERO-SHOT RESULTS — SQLi-only subset of CSIC 2010
1,389 SQLi attacks vs 56,000 benign | Best F1 (fine-grained scan, step=0.01)
Model          Mode         Strategy                  Thresh   Recall   FP/10k  Precision       F1
------------------------------------------------------------------------------------------
MobileBERT     zero-shot    Way 3 (cssupport)           0.99   0.7991   210.54     0.4849   0.6036

Saved: /home/ubuntu/webattack-detector/notebooks/results/metrics/17_mobilebert_sqli_only_summary.csv
Saved: /home/ubuntu/webattack-detector/notebooks/results/metrics/17_mobilebert_sqli_only_summary.json


## 8. Cost Analysis

Measures MobileBERT latency and throughput on the T4 GPU.
Results feed into the master comparison notebook alongside NB18 RF cost numbers.

In [9]:
import time
import torch
import torch.nn.functional as F
import urllib.parse
import numpy as np
from transformers import MobileBertTokenizer, MobileBertForSequenceClassification

# ── 10 mixed SQLi access log entries ─────────────────────────────
# Real Apache Combined Log Format
# Mix of GET and POST, different SQLi families, benign-looking paths
ACCESS_LOG_ENTRIES = [
    # Boolean-based
    '192.168.1.1 - - [25/Jun/2024:10:01:01 +0000] "GET /shop/product?id=1\'+OR+\'1\'=\'1&Submit=Submit HTTP/1.1" 200 1234 "-" "Mozilla/5.0"',
    # UNION-based
    '192.168.1.2 - - [25/Jun/2024:10:01:02 +0000] "GET /search?q=test\'+UNION+SELECT+username,password+FROM+users-- HTTP/1.1" 200 5678 "-" "Mozilla/5.0"',
    # Time-based blind
    '192.168.1.3 - - [25/Jun/2024:10:01:03 +0000] "GET /item?id=1\';+waitfor+delay+\'0:0:5\';-- HTTP/1.1" 200 2345 "-" "Mozilla/5.0"',
    # Stacked queries
    '192.168.1.4 - - [25/Jun/2024:10:01:04 +0000] "GET /user?login=admin\';+DROP+TABLE+users;-- HTTP/1.1" 200 3456 "-" "Mozilla/5.0"',
    # pg_sleep (PostgreSQL time-based)
    '192.168.1.5 - - [25/Jun/2024:10:01:05 +0000] "GET /filter?cat=shoes\'+AND+1=CAST((SELECT+pg_sleep(5))+AS+int)-- HTTP/1.1" 200 4567 "-" "Mozilla/5.0"',
    # OR string comparison bypass
    '192.168.1.6 - - [25/Jun/2024:10:01:06 +0000] "GET /auth?user=admin\'+OR+\'a\'=\'a HTTP/1.1" 200 5678 "-" "Mozilla/5.0"',
    # Schema probing
    '192.168.1.7 - - [25/Jun/2024:10:01:07 +0000] "GET /api?q=1+AND+1=1+UNION+SELECT+table_name+FROM+information_schema.tables-- HTTP/1.1" 200 6789 "-" "Mozilla/5.0"',
    # Double-encoded
    '192.168.1.8 - - [25/Jun/2024:10:01:08 +0000] "GET /view?id=1%27%20OR%20%271%27%3D%271 HTTP/1.1" 200 7890 "-" "Mozilla/5.0"',
    # Comment-based
    '192.168.1.9 - - [25/Jun/2024:10:01:09 +0000] "GET /page?id=1/**/OR/**/1=1-- HTTP/1.1" 200 8901 "-" "Mozilla/5.0"',
    # Benchmark (MySQL)
    '192.168.1.10 - - [25/Jun/2024:10:01:10 +0000] "GET /sort?by=name\'+AND+BENCHMARK(5000000,MD5(1))-- HTTP/1.1" 200 9012 "-" "Mozilla/5.0"',
]


# ── Access log parser ─────────────────────────────────────────────
def parse_access_log_entry(log_line):
    """
    Extract query string values from an Apache Combined Log Format line.
    Returns the decoded query parameter values joined as a string,
    matching extract_query_values() used during evaluation.
    """
    try:
        # Extract the quoted request part: "METHOD /path?qs HTTP/1.1"
        parts     = log_line.split('"')
        request   = parts[1]                          # e.g. GET /path?qs HTTP/1.1
        url_part  = request.split(' ')[1]             # e.g. /path?qs
        parsed    = urllib.parse.urlparse(url_part)
        qs        = parsed.query
        if not qs:
            return urllib.parse.unquote(url_part)     # fallback: full path
        params = urllib.parse.parse_qs(qs, keep_blank_values=False)
        values = [
            urllib.parse.unquote(v).strip()
            for vlist in params.values()
            for v in vlist
            if urllib.parse.unquote(v).strip()
        ]
        return ' '.join(values) if values else urllib.parse.unquote(url_part)
    except Exception as e:
        return None


# ── Parse all 10 entries ──────────────────────────────────────────
parsed_queries = []
print('Parsed access log entries:')
print('─' * 70)
for i, line in enumerate(ACCESS_LOG_ENTRIES, 1):
    q = parse_access_log_entry(line)
    parsed_queries.append(q)
    print(f'[{i:>2}] {q[:80] if q else "FAILED TO PARSE"}')

valid_queries = [q for q in parsed_queries if q]
print(f'\nSuccessfully parsed: {len(valid_queries)}/{len(ACCESS_LOG_ENTRIES)}')


# ── Load models for cost measurement ─────────────────────────────
# We measure cost for: zero-shot MobileBERT, fine-tuned MobileBERT (best strategy)
# Load from saved checkpoints — no retraining

print('\nLoading models for cost measurement...')

# Zero-shot
zs_path  = f'{MODELS_DIR}/15_mobilebert_zeroshot'
tok_cost  = MobileBertTokenizer.from_pretrained('google/mobilebert-uncased')
model_zs_cost = MobileBertForSequenceClassification.from_pretrained(zs_path)
model_zs_cost.to(DEVICE)
model_zs_cost.eval()
print(f'  ✅ Zero-shot model loaded from {zs_path}')

# Fine-tuned best (query values — load whichever finished successfully)
# Update path if a different strategy produced valid results
ft_path = f'{MODELS_DIR}/17_mobilebert_query_values'
model_ft_cost = MobileBertForSequenceClassification.from_pretrained(ft_path)
model_ft_cost.to(DEVICE)
model_ft_cost.eval()
print(f'  ✅ Fine-tuned model loaded from {ft_path}')


# ── Single request latency (100 repetitions per request) ──────────
def measure_single_request_latency(model, tokenizer, queries,
                                   n_repeats=100, device='cpu'):
    """
    Measure latency for processing one request at a time.
    Simulates real WAF: each HTTP request scored individually.
    Returns per-request latency in milliseconds.
    """
    latencies = []
    model.eval()
    with torch.no_grad():
        for query in queries:
            for _ in range(n_repeats):
                t0     = time.perf_counter()
                inputs = tokenizer(
                    query, truncation=True, padding='max_length',
                    max_length=MAX_LENGTH, return_tensors='pt'
                ).to(device)
                logits = model(**inputs).logits
                probs  = torch.softmax(logits, dim=1)[:, 1].item()
                t1     = time.perf_counter()
                latencies.append((t1 - t0) * 1000)   # ms

    return np.array(latencies)


# ── Throughput (requests per second at different batch sizes) ─────
def measure_throughput(model, tokenizer, queries,
                       batch_sizes=(1, 10, 50, 100), device='cpu'):
    """
    Measure throughput at different batch sizes.
    Repeats each batch 20 times for stable measurement.
    Returns dict: batch_size -> requests_per_second.
    """
    results = {}
    model.eval()
    with torch.no_grad():
        for bs in batch_sizes:
            batch_queries = (queries * ((bs // len(queries)) + 1))[:bs]
            times = []
            for _ in range(20):
                t0     = time.perf_counter()
                inputs = tokenizer(
                    batch_queries, truncation=True, padding=True,
                    max_length=MAX_LENGTH, return_tensors='pt'
                ).to(device)
                logits = model(**inputs).logits
                probs  = torch.softmax(logits, dim=1)[:, 1].cpu().numpy()
                t1     = time.perf_counter()
                times.append(t1 - t0)
            avg_time   = np.mean(times)
            throughput = bs / avg_time
            results[bs] = throughput
    return results


# ── GPU warmup (prevents first-call overhead skewing results) ─────
print('\nWarming up GPU...')
warmup_inputs = tok_cost(
    ['warmup query'] * 10, truncation=True, padding='max_length',
    max_length=MAX_LENGTH, return_tensors='pt'
).to(DEVICE)
with torch.no_grad():
    for _ in range(10):
        _ = model_zs_cost(**warmup_inputs).logits
print('  GPU warmed up.')


# ── RUN: Single request latency ───────────────────────────────────
N_REPEATS = 100
print(f'\nMeasuring single-request latency ({N_REPEATS} repeats per query)...')

lat_zs = measure_single_request_latency(
    model_zs_cost, tok_cost, valid_queries, N_REPEATS, DEVICE
)
lat_ft = measure_single_request_latency(
    model_ft_cost, tok_cost, valid_queries, N_REPEATS, DEVICE
)

print('\n' + '─' * 60)
print('SINGLE REQUEST LATENCY (ms) — GPU inference, 1 request at a time')
print('─' * 60)
print(f'{"Model":<35} {"Mean":>8} {"Std":>8} {"P50":>8} {"P95":>8} {"P99":>8}')
print('─' * 60)
for label, lat in [
    ('MobileBERT zero-shot (cssupport)',  lat_zs),
    ('MobileBERT fine-tuned (Way 3)',     lat_ft),
]:
    print(f'{label:<35} {np.mean(lat):>8.2f} {np.std(lat):>8.2f} '
          f'{np.percentile(lat,50):>8.2f} {np.percentile(lat,95):>8.2f} '
          f'{np.percentile(lat,99):>8.2f}')
print('─' * 60)


# ── RUN: Throughput ───────────────────────────────────────────────
BATCH_SIZES = (1, 10, 50, 100)
print(f'\nMeasuring throughput at batch sizes {BATCH_SIZES}...')

tput_zs = measure_throughput(model_zs_cost, tok_cost, valid_queries, BATCH_SIZES, DEVICE)
tput_ft = measure_throughput(model_ft_cost, tok_cost, valid_queries, BATCH_SIZES, DEVICE)

print('\n' + '─' * 65)
print('THROUGHPUT (requests/second) — GPU inference')
print('─' * 65)
print(f'{"Batch Size":>12}  {"MobileBERT zero-shot":>22}  {"MobileBERT fine-tuned":>22}')
print('─' * 65)
for bs in BATCH_SIZES:
    print(f'{bs:>12}  {tput_zs[bs]:>22.1f}  {tput_ft[bs]:>22.1f}')
print('─' * 65)


# ── Model size ────────────────────────────────────────────────────
def model_size_mb(model):
    total = sum(p.numel() * p.element_size() for p in model.parameters())
    return total / (1024 ** 2)

def count_params(model):
    return sum(p.numel() for p in model.parameters())

print('\n' + '─' * 60)
print('MODEL SIZE')
print('─' * 60)
for label, model in [
    ('MobileBERT zero-shot', model_zs_cost),
    ('MobileBERT fine-tuned', model_ft_cost),
]:
    print(f'{label:<30} {model_size_mb(model):>8.1f} MB  '
          f'{count_params(model)/1e6:>6.1f}M params')
print('─' * 60)


# ── AWS cost estimate ─────────────────────────────────────────────
# g4dn.xlarge (T4 GPU): ~$0.526/hr on-demand (us-east-1, May 2026)
# c5.xlarge   (CPU):    ~$0.170/hr on-demand (us-east-1, May 2026)
GPU_COST_PER_HR  = 0.526
CPU_COST_PER_HR  = 0.170
REQUESTS_PER_DAY = 1_000_000   # 1M requests/day — typical medium web app

# GPU: use throughput at batch_size=1 (real-time WAF, no batching)
rps_zs    = tput_zs[1]
rps_ft    = tput_ft[1]

# Hours of GPU time needed per day at batch_size=1
hrs_zs    = (REQUESTS_PER_DAY / rps_zs) / 3600
hrs_ft    = (REQUESTS_PER_DAY / rps_ft) / 3600

cost_zs_gpu = hrs_zs * GPU_COST_PER_HR
cost_ft_gpu = hrs_ft * GPU_COST_PER_HR

# RF on CPU: ~0.1ms per request (approximate — update with actual RF timing)
RF_LATENCY_MS     = 0.1
rps_rf            = 1000 / RF_LATENCY_MS
hrs_rf            = (REQUESTS_PER_DAY / rps_rf) / 3600
cost_rf_cpu       = hrs_rf * CPU_COST_PER_HR

print('\n' + '─' * 70)
print(f'AWS COST ESTIMATE — {REQUESTS_PER_DAY:,} requests/day')
print('Assumes real-time WAF (batch_size=1, no request queuing)')
print('─' * 70)
print(f'{"Model":<35} {"Infra":>8} {"RPS":>10} {"Hrs/day":>10} {"$/day":>10}')
print('─' * 70)
print(f'{"RF (estimated)":<35} {"CPU":>8} {rps_rf:>10.0f} {hrs_rf:>10.4f} {cost_rf_cpu:>10.4f}')
print(f'{"MobileBERT zero-shot":<35} {"GPU":>8} {rps_zs:>10.1f} {hrs_zs:>10.2f} {cost_zs_gpu:>10.4f}')
print(f'{"MobileBERT fine-tuned (Way 3)":<35} {"GPU":>8} {rps_ft:>10.1f} {hrs_ft:>10.2f} {cost_ft_gpu:>10.4f}')
print('─' * 70)
print(f'\nNote: RF cost uses estimated latency of {RF_LATENCY_MS}ms — update with actual')
print(f'measured RF latency from your RF evaluation notebook.')
print(f'GPU pricing: g4dn.xlarge ${GPU_COST_PER_HR}/hr (us-east-1 on-demand)')
print(f'CPU pricing: c5.xlarge   ${CPU_COST_PER_HR}/hr (us-east-1 on-demand)')


# ── Save cost results ─────────────────────────────────────────────
cost_summary = {
    'test_queries': len(valid_queries),
    'n_repeats_latency': N_REPEATS,
    'latency_ms': {
        'mobilebert_zeroshot': {
            'mean': round(float(np.mean(lat_zs)), 3),
            'std':  round(float(np.std(lat_zs)), 3),
            'p50':  round(float(np.percentile(lat_zs, 50)), 3),
            'p95':  round(float(np.percentile(lat_zs, 95)), 3),
            'p99':  round(float(np.percentile(lat_zs, 99)), 3),
        },
        'mobilebert_finetuned': {
            'mean': round(float(np.mean(lat_ft)), 3),
            'std':  round(float(np.std(lat_ft)), 3),
            'p50':  round(float(np.percentile(lat_ft, 50)), 3),
            'p95':  round(float(np.percentile(lat_ft, 95)), 3),
            'p99':  round(float(np.percentile(lat_ft, 99)), 3),
        },
    },
    'throughput_rps': {
        'mobilebert_zeroshot':  {str(bs): round(v, 1) for bs, v in tput_zs.items()},
        'mobilebert_finetuned': {str(bs): round(v, 1) for bs, v in tput_ft.items()},
    },
    'model_size_mb': {
        'mobilebert_zeroshot':  round(model_size_mb(model_zs_cost), 1),
        'mobilebert_finetuned': round(model_size_mb(model_ft_cost), 1),
    },
    'aws_cost_per_day_usd': {
        'rf_cpu_estimated':         round(cost_rf_cpu, 4),
        'mobilebert_zeroshot_gpu':  round(cost_zs_gpu, 4),
        'mobilebert_finetuned_gpu': round(cost_ft_gpu, 4),
    },
}

with open(f'{RESULTS_DIR}/17_cost_analysis.json', 'w') as f:
    json.dump(cost_summary, f, indent=2)
print(f'\nSaved: {RESULTS_DIR}/17_cost_analysis.json')

# Cleanup
del model_zs_cost, model_ft_cost
torch.cuda.empty_cache()
print('Models unloaded from GPU.')

Parsed access log entries:
──────────────────────────────────────────────────────────────────────
[ 1] 1' OR '1'='1 Submit
[ 2] test' UNION SELECT username,password FROM users--
[ 3] 1'; waitfor delay '0:0:5';--
[ 4] admin'; DROP TABLE users;--
[ 5] shoes' AND 1=CAST((SELECT pg_sleep(5)) AS int)--
[ 6] admin' OR 'a'='a
[ 7] 1 AND 1=1 UNION SELECT table_name FROM information_schema.tables--
[ 8] 1' OR '1'='1
[ 9] 1/**/OR/**/1=1--
[10] name' AND BENCHMARK(5000000,MD5(1))--

Successfully parsed: 10/10

Loading models for cost measurement...


Loading weights:   0%|          | 0/1113 [00:00<?, ?it/s]

  ✅ Zero-shot model loaded from /home/ubuntu/webattack-detector/notebooks/results/models/15_mobilebert_zeroshot


Loading weights:   0%|          | 0/1113 [00:00<?, ?it/s]

  ✅ Fine-tuned model loaded from /home/ubuntu/webattack-detector/notebooks/results/models/17_mobilebert_query_values

Warming up GPU...
  GPU warmed up.

Measuring single-request latency (100 repeats per query)...

────────────────────────────────────────────────────────────
SINGLE REQUEST LATENCY (ms) — GPU inference, 1 request at a time
────────────────────────────────────────────────────────────
Model                                   Mean      Std      P50      P95      P99
────────────────────────────────────────────────────────────
MobileBERT zero-shot (cssupport)       29.90     0.90    29.85    30.15    30.53
MobileBERT fine-tuned (Way 3)          29.97     0.17    29.99    30.21    30.32
────────────────────────────────────────────────────────────

Measuring throughput at batch sizes (1, 10, 50, 100)...

─────────────────────────────────────────────────────────────────
THROUGHPUT (requests/second) — GPU inference
────────────────────────────────────────────────────────────────